# Jamaica Mangrove Assessment: Global Mangrove Watch vs Forces of Nature

This notebook:
1. Reads the global mangrove vector layer (`gmw_v3_2020_vec.shp`)
2. Clips it to Jamaica + coastal buffer (to avoid losing nearshore mangroves)
3. Reads Forces of Nature (FN) mangroves
4. Compares extent and location differences between the two layers

Outputs include summary area metrics and overlay maps.


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from shapely.geometry import GeometryCollection

plt.style.use('default')
pd.set_option('display.max_columns', 100)

import matplotlib.patches as mpatches


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

# Input paths
# Optional alternative: gmw_v3_f1996_t2020_vec/gmw_v3_f1996_t2020_vec.shp
gmw_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/state_of_art_global_mangrove_layer_bunting_2022/gmw_v3_2020_vec/gmw_v3_2020_vec.shp'
fn_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

# Coastal clip settings (meters in EPSG:3448)
COASTAL_BUFFER_M = 5000
FN_EXPAND_BUFFER_M = 2000

# Output settings
output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/mangrove_assessment'
output_dir.mkdir(parents=True, exist_ok=True)
SAVE_OUTPUTS = False

print('Working dir:', Path.cwd())
print('Project root:', ROOT)
for p in [gmw_path, fn_path, jamaica_boundary_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


In [ ]:
# Read Jamaica boundary and FN mangroves in Jamaica CRS (EPSG:3448)
jamaica = gpd.read_file(jamaica_boundary_path).to_crs(3448)
fn = gpd.read_file(fn_path).to_crs(3448)

jamaica = jamaica[jamaica.geometry.notnull() & ~jamaica.geometry.is_empty].copy()
fn = fn[fn.geometry.notnull() & ~fn.geometry.is_empty].copy()

print('Jamaica boundary features:', len(jamaica))
print('FN mangrove features:', len(fn))
print('FN area (ha):', round(fn.geometry.area.sum() / 10_000, 2))


In [ ]:
# Build a coastal clip zone to avoid excluding nearshore mangroves
jamaica_union = unary_union(jamaica.geometry.tolist())
fn_union = unary_union(fn.geometry.tolist())

clip_geom_3448 = jamaica_union.buffer(COASTAL_BUFFER_M).union(fn_union.buffer(FN_EXPAND_BUFFER_M))
clip_zone_3448 = gpd.GeoDataFrame(geometry=[clip_geom_3448], crs=3448)

# Read GMW using bbox in native GMW CRS for speed
gmw_sample = gpd.read_file(gmw_path, rows=1)
gmw_crs = gmw_sample.crs
clip_zone_gmw = clip_zone_3448.to_crs(gmw_crs)

bbox = tuple(clip_zone_gmw.total_bounds)
gmw_candidates = gpd.read_file(gmw_path, bbox=bbox)

# Precise clip in GMW CRS, then convert to EPSG:3448 for comparison
gmw_clipped = gpd.overlay(gmw_candidates, clip_zone_gmw, how='intersection')
gmw = gmw_clipped.to_crs(3448)
gmw = gmw[gmw.geometry.notnull() & ~gmw.geometry.is_empty].copy()

print('GMW candidate features in bbox:', len(gmw_candidates))
print('GMW clipped features:', len(gmw))
print('GMW clipped area (ha):', round(gmw.geometry.area.sum() / 10_000, 2))


In [ ]:
# Compare extents using dissolved geometries

gmw_union = unary_union(gmw.geometry.tolist()) if len(gmw) else GeometryCollection()
fn_union = unary_union(fn.geometry.tolist()) if len(fn) else GeometryCollection()

overlap_geom = gmw_union.intersection(fn_union)
gmw_only_geom = gmw_union.difference(fn_union)
fn_only_geom = fn_union.difference(gmw_union)
union_geom = gmw_union.union(fn_union)

metrics = {
    'gmw_area_ha': gmw_union.area / 10_000,
    'fn_area_ha': fn_union.area / 10_000,
    'overlap_area_ha': overlap_geom.area / 10_000,
    'gmw_only_area_ha': gmw_only_geom.area / 10_000,
    'fn_only_area_ha': fn_only_geom.area / 10_000,
    'union_area_ha': union_geom.area / 10_000,
    'jaccard_overlap_ratio': (overlap_geom.area / union_geom.area) if union_geom.area > 0 else 0.0,
    'overlap_pct_of_fn': (overlap_geom.area / fn_union.area * 100) if fn_union.area > 0 else 0.0,
    'overlap_pct_of_gmw': (overlap_geom.area / gmw_union.area * 100) if gmw_union.area > 0 else 0.0,
}

metrics_df = pd.DataFrame([metrics]).round(4)
metrics_df


In [ ]:
# Quick overlay map: GMW vs FN
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

jamaica.boundary.plot(ax=ax, color='black', linewidth=0.6, alpha=0.7)
if len(gmw) > 0:
    gmw.plot(ax=ax, color='#2b8cbe', alpha=0.45, edgecolor='none')
if len(fn) > 0:
    fn.plot(ax=ax, color='#f03b20', alpha=0.45, edgecolor='none')

ax.set_title('Jamaica Mangroves: Global Mangrove Watch vs FN')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#2b8cbe', edgecolor='none', alpha=0.45, label='GMW (clipped)'),
    mpatches.Patch(facecolor='#f03b20', edgecolor='none', alpha=0.45, label='FN mangroves'),
    mpatches.Patch(facecolor='none', edgecolor='black', linewidth=1.0, label='Jamaica boundary'),
]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.95)

if SAVE_OUTPUTS:
    out_png = output_dir / 'gmw_vs_fn_overlay.png'
    fig.savefig(out_png, dpi=300)
    print('Saved:', out_png)
else:
    print('PNG export skipped (SAVE_OUTPUTS=False)')

plt.show()


In [ ]:
# Difference map: overlap, GMW-only, FN-only
layers = [
    ('Overlap', overlap_geom, '#31a354'),
    ('GMW only', gmw_only_geom, '#2b8cbe'),
    ('FN only', fn_only_geom, '#f03b20'),
]

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
jamaica.boundary.plot(ax=ax, color='black', linewidth=0.6, alpha=0.7)

for name, geom, color in layers:
    if geom is not None and not geom.is_empty:
        gpd.GeoDataFrame({'label': [name]}, geometry=[geom], crs=3448).plot(
            ax=ax,
            color=color,
            alpha=0.55,
            edgecolor='none',
        )

ax.set_title('Mangrove Layer Comparison: Overlap and Differences')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#31a354', edgecolor='none', alpha=0.55, label='Overlap'),
    mpatches.Patch(facecolor='#2b8cbe', edgecolor='none', alpha=0.55, label='GMW only'),
    mpatches.Patch(facecolor='#f03b20', edgecolor='none', alpha=0.55, label='FN only'),
    mpatches.Patch(facecolor='none', edgecolor='black', linewidth=1.0, label='Jamaica boundary'),
]
ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.95)

if SAVE_OUTPUTS:
    diff_png = output_dir / 'gmw_vs_fn_overlap_difference_map.png'
    fig.savefig(diff_png, dpi=300)
    print('Saved:', diff_png)
else:
    print('PNG export skipped (SAVE_OUTPUTS=False)')

plt.show()


In [ ]:
# Optional exports
if SAVE_OUTPUTS:
    # Save clipped GMW and FN for reproducibility
    gmw_out = output_dir / 'gmw_clipped_jamaica_coastal_buffer.gpkg'
    fn_out = output_dir / 'fn_mangroves_for_comparison.gpkg'
    metrics_out = output_dir / 'gmw_vs_fn_comparison_metrics.csv'

    gmw.to_file(gmw_out, driver='GPKG')
    fn.to_file(fn_out, driver='GPKG')
    metrics_df.to_csv(metrics_out, index=False)

    print('Saved:', gmw_out)
    print('Saved:', fn_out)
    print('Saved:', metrics_out)
else:
    print('Vector/CSV export skipped (SAVE_OUTPUTS=False)')


## Notes
- The clip zone uses a coastal buffer around Jamaica plus a small FN expansion buffer.
- Adjust `COASTAL_BUFFER_M` and `FN_EXPAND_BUFFER_M` in the config cell if needed.
- Area comparisons are calculated in EPSG:3448 (meters), then converted to hectares.


In [ ]:
# Summary statistics (requested overlap/non-overlap metrics)
# Areas are in hectares.

gmw_area_ha = metrics['gmw_area_ha']
fn_area_ha = metrics['fn_area_ha']
overlap_area_ha = metrics['overlap_area_ha']
gmw_only_area_ha = metrics['gmw_only_area_ha']
fn_only_area_ha = metrics['fn_only_area_ha']
union_area_ha = metrics['union_area_ha']

summary_rows = [
    {
        'metric': 'Total overlap area',
        'area_ha': overlap_area_ha,
        'fn_total_area_ha': fn_area_ha,
        'gmw_total_area_ha': gmw_area_ha,
        'pct_of_FN': (overlap_area_ha / fn_area_ha * 100) if fn_area_ha > 0 else 0,
        'pct_of_GMW': (overlap_area_ha / gmw_area_ha * 100) if gmw_area_ha > 0 else 0,
        'pct_of_combined_union': (overlap_area_ha / union_area_ha * 100) if union_area_ha > 0 else 0,
    },
    {
        'metric': 'FN area not overlapped (FN only)',
        'area_ha': fn_only_area_ha,
        'fn_total_area_ha': fn_area_ha,
        'gmw_total_area_ha': gmw_area_ha,
        'pct_of_FN': (fn_only_area_ha / fn_area_ha * 100) if fn_area_ha > 0 else 0,
        'pct_of_GMW': (fn_only_area_ha / gmw_area_ha * 100) if gmw_area_ha > 0 else 0,
        'pct_of_combined_union': (fn_only_area_ha / union_area_ha * 100) if union_area_ha > 0 else 0,
    },
    {
        'metric': 'GMW area not overlapped (GMW only)',
        'area_ha': gmw_only_area_ha,
        'fn_total_area_ha': fn_area_ha,
        'gmw_total_area_ha': gmw_area_ha,
        'pct_of_FN': (gmw_only_area_ha / fn_area_ha * 100) if fn_area_ha > 0 else 0,
        'pct_of_GMW': (gmw_only_area_ha / gmw_area_ha * 100) if gmw_area_ha > 0 else 0,
        'pct_of_combined_union': (gmw_only_area_ha / union_area_ha * 100) if union_area_ha > 0 else 0,
    },
]

summary_stats_df = pd.DataFrame(summary_rows)
summary_stats_df[[
    'area_ha', 'fn_total_area_ha', 'gmw_total_area_ha',
    'pct_of_FN', 'pct_of_GMW', 'pct_of_combined_union'
]] = summary_stats_df[[
    'area_ha', 'fn_total_area_ha', 'gmw_total_area_ha',
    'pct_of_FN', 'pct_of_GMW', 'pct_of_combined_union'
]].round(3)

print('Requested comparison summary:')
display(summary_stats_df)

# Extra headline percentages
headline = pd.DataFrame([
    {
        'Total mangrove area (union, ha)': round(union_area_ha, 3),
        'FN total area (ha)': round(fn_area_ha, 3),
        'GMW total area (ha)': round(gmw_area_ha, 3),
        'FN overlap %': round((overlap_area_ha / fn_area_ha * 100) if fn_area_ha > 0 else 0, 3),
        'FN not-overlap %': round((fn_only_area_ha / fn_area_ha * 100) if fn_area_ha > 0 else 0, 3),
        'GMW overlap %': round((overlap_area_ha / gmw_area_ha * 100) if gmw_area_ha > 0 else 0, 3),
        'GMW not-overlap %': round((gmw_only_area_ha / gmw_area_ha * 100) if gmw_area_ha > 0 else 0, 3),
        'Jaccard overlap % (overlap/union)': round((overlap_area_ha / union_area_ha * 100) if union_area_ha > 0 else 0, 3),
    }
])

display(headline)
